In [23]:
import pandas as pd
import numpy as np
import shap
import joblib

df_clean = pd.read_csv("data/processed/cleaned_clients.csv")
# Load the saved model bundle
model_bundle = joblib.load("models/churn_model_bundle.joblib")

# Restore the model and threshold
final_xgb_model = model_bundle["model"]
selected_threshold = model_bundle["threshold"]

print("Model loaded successfully!")
print("Threshold:", selected_threshold)

In [25]:
import pandas as pd
import numpy as np

# Load cleaned dataset
df = pd.read_csv(
    "data/processed/cleaned_clients.csv"
)

# Load saved customer risk predictions
customer_risk_df = pd.read_csv(
    "../data/processed/customer_risk_predictions.csv",
    index_col="original_index"
)

# Restore original index as integer
customer_risk_df.index = customer_risk_df.index.astype(int)

print("Dataset shape:", df.shape)
print("Risk predictions:", customer_risk_df.shape)

display(customer_risk_df.head())

In [27]:
from sklearn.model_selection import train_test_split

# Recreate feature engineering
df["TenureGroup"] = pd.cut(
    df["tenure"],
    bins=[-1, 6, 12, 24, 48, 72],
    labels=[
        "0-6 months",
        "7-12 months",
        "13-24 months",
        "25-48 months",
        "49-72 months"
    ]
)

df["AvgMonthlySpend"] = np.where(
    df["tenure"] > 0,
    df["TotalCharges"] / df["tenure"],
    df["MonthlyCharges"]
)

service_columns = [
    "PhoneService",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

df["ServiceCount"] = (
    df[service_columns]
    .replace({
        "Yes": 1,
        "No": 0,
        "No internet service": 0,
        "No phone service": 0
    })
    .sum(axis=1)
)

# Separate features and target
X = df.drop(columns=["Churn", "customerID"])

y = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

# Same original train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_test shape:", X_test.shape)
print("X_test index matches risk data:",
      X_test.index.equals(customer_risk_df.index))

In [28]:
monthly_charge_limit = X_train[
    "MonthlyCharges"
].quantile(0.75)

print("Monthly charge benchmark:", monthly_charge_limit)

In [30]:
def generate_retention_recommendations(row):
    """Create customer-specific retention actions from a feature row."""
    recommendations = []

    contract = str(row.get("Contract", "")).strip().lower()
    tenure = pd.to_numeric(row.get("tenure", 0), errors="coerce")
    monthly_charges = pd.to_numeric(
        row.get("MonthlyCharges", np.nan), errors="coerce"
    )

    if "month-to-month" in contract:
        recommendations.append(
            "Offer an incentive to move to a one-year contract, such as a bill credit or price guarantee."
        )

    if pd.notna(tenure) and tenure <= 12:
        recommendations.append(
            "Provide onboarding support and check in proactively during the first year."
        )

    if (
        pd.notna(monthly_charges)
        and "monthly_charge_limit" in globals()
        and monthly_charges > monthly_charge_limit
    ):
        recommendations.append(
            "Review the current plan and offer a lower-cost package or a tailored discount."
        )

    for column, action in [
        ("OnlineSecurity", "Offer an Online Security trial or discounted add-on."),
        ("TechSupport", "Offer a Tech Support trial or discounted add-on."),
    ]:
        value = str(row.get(column, "")).strip().lower()
        if value == "no":
            recommendations.append(action)

    if not recommendations:
        recommendations.append(
            "Maintain regular customer engagement and offer a personalized loyalty benefit."
        )

    return recommendations


recommendation_records = []

for idx, row in X_test.iterrows():
    risk = customer_risk_df.loc[idx, "Risk_Category"]
    recommendations = generate_retention_recommendations(row)

    recommendation_records.append({
        "CustomerID": df.loc[idx, "customerID"],
        "Churn_Probability": customer_risk_df.loc[
            idx, "Churn_Probability"
        ],
        "Risk_Category": risk,
        "Recommendations": recommendations
    })

recommendation_df = pd.DataFrame(recommendation_records)

print("Recommendation records created:", len(recommendation_df))
display(recommendation_df.head())

In [32]:
from collections import Counter

recommendation_counts = Counter()

for recommendations in recommendation_df["Recommendations"]:
    for recommendation in recommendations:
        # Recommendations are stored as strings, not dictionaries.
        recommendation_counts[recommendation] += 1

recommendation_summary = pd.DataFrame(
    recommendation_counts.items(),
    columns=["Recommendation_Reason", "Customer_Count"]
).sort_values(
    "Customer_Count",
    ascending=False
).reset_index(drop=True)

display(recommendation_summary)

In [34]:
required_variables = [
    "final_xgb_model",
    "X_test",
    "df_clean",
    "model_bundle"
]

for variable in required_variables:
    print(f"{variable}: {variable in globals()}")

In [36]:
import matplotlib.pyplot as plt

recommendation_summary.plot(
    x="Recommendation_Reason",
    y="Customer_Count",
    kind="barh",
    figsize=(10, 6),
    legend=False
)

plt.title("Retention Recommendation Distribution")
plt.xlabel("Number of Customers")
plt.ylabel("Recommendation Reason")
plt.tight_layout()
plt.show()

In [38]:
recommendation_details = []

# Map the customer's risk category to an action priority. Recommendation entries
# in this notebook are currently strings, but this also supports dictionaries.
priority_by_risk = {
    "HIGH": "High",
    "MEDIUM": "Medium",
    "LOW": "Low"
}

for _, row in recommendation_df.iterrows():
    recommendations = row["Recommendations"]

    # Ensure a single string recommendation is treated as one item.
    if isinstance(recommendations, str):
        recommendations = [recommendations]
    elif not isinstance(recommendations, (list, tuple)):
        recommendations = [] if pd.isna(recommendations) else [recommendations]

    for item in recommendations:
        if isinstance(item, dict):
            # Support the structured format if recommendations are changed later.
            reason = item.get("Reason", item.get("reason", ""))
            action = item.get("Action", item.get("action", reason))
            priority = item.get(
                "Priority",
                item.get("priority", priority_by_risk.get(str(row["Risk_Category"]).upper(), "Low"))
            )
        else:
            # Current recommendations are strings, each describing the action.
            reason = str(item)
            action = str(item)
            priority = priority_by_risk.get(str(row["Risk_Category"]).upper(), "Low")

        recommendation_details.append({
            "CustomerID": row["CustomerID"],
            "Risk_Category": row["Risk_Category"],
            "Churn_Probability": row["Churn_Probability"],
            "Recommendation_Reason": reason,
            "Recommended_Action": action,
            "Priority": priority
        })

recommendation_details_df = pd.DataFrame(
    recommendation_details,
    columns=[
        "CustomerID",
        "Risk_Category",
        "Churn_Probability",
        "Recommendation_Reason",
        "Recommended_Action",
        "Priority"
    ]
)

print("Total recommendation records:", len(recommendation_details_df))
display(recommendation_details_df.head(10))

In [40]:
risk_recommendation_summary = pd.crosstab(
    recommendation_details_df["Recommendation_Reason"],
    recommendation_details_df["Risk_Category"]
)

display(
    risk_recommendation_summary
)

In [42]:
import matplotlib.pyplot as plt

risk_recommendation_summary.plot(
    kind="barh",
    figsize=(11, 7)
)

plt.title("Retention Recommendations by Customer Risk")
plt.xlabel("Number of Customers")
plt.ylabel("Recommendation Reason")
plt.tight_layout()
plt.show()

In [43]:
def assign_action_urgency(row):

    if row["Risk_Category"] == "HIGH":
        return "URGENT"

    elif row["Risk_Category"] == "MEDIUM":
        return "PLANNED"

    else:
        return "MONITOR"


recommendation_details_df["Action_Urgency"] = (
    recommendation_details_df.apply(
        assign_action_urgency,
        axis=1
    )
)

display(
    recommendation_details_df[
        [
            "CustomerID",
            "Risk_Category",
            "Recommendation_Reason",
            "Priority",
            "Action_Urgency"
        ]
    ].head(15)
)

In [45]:
!pip install shap==0.54.1

In [47]:
import pandas as pd
import numpy as np
import shap

# Prepare the data in the same feature space consumed by the tree estimator.
# If final_xgb_model is a scikit-learn Pipeline, apply its preprocessing steps
# and explain the final estimator; predictions below still use the full pipeline.
model_for_shap = final_xgb_model
X_for_shap = X_test.copy()
feature_names = list(X_test.columns)

if hasattr(final_xgb_model, "steps") and final_xgb_model.steps:
    model_for_shap = final_xgb_model.steps[-1][1]

    if len(final_xgb_model.steps) > 1:
        preprocessing = final_xgb_model[:-1]
        X_for_shap = preprocessing.transform(X_test)

        try:
            feature_names = list(preprocessing.get_feature_names_out())
        except (AttributeError, TypeError, ValueError):
            if hasattr(X_for_shap, "columns"):
                feature_names = list(X_for_shap.columns)
            elif hasattr(model_for_shap, "feature_names_in_"):
                feature_names = list(model_for_shap.feature_names_in_)
            else:
                feature_names = [f"feature_{i}" for i in range(X_for_shap.shape[1])]

# Use the tree explainer for the final XGBoost/tree estimator.
explainer = shap.TreeExplainer(model_for_shap)
shap_result = explainer.shap_values(X_for_shap)

# SHAP versions may return a list (one array per class), an Explanation, or an
# array with a final class dimension. For churn, retain the positive class.
if hasattr(shap_result, "values"):
    shap_array = np.asarray(shap_result.values)
else:
    shap_array = shap_result

if isinstance(shap_array, list):
    shap_array = np.asarray(shap_array[1] if len(shap_array) > 1 else shap_array[0])
else:
    shap_array = np.asarray(shap_array)
    if shap_array.ndim == 3 and shap_array.shape[-1] > 1:
        shap_array = shap_array[:, :, 1]

if shap_array.ndim != 2:
    raise ValueError(f"Expected 2D SHAP values, got shape {shap_array.shape}")

# Align feature names with the columns actually returned by the estimator.
if len(feature_names) != shap_array.shape[1]:
    feature_names = [f"feature_{i}" for i in range(shap_array.shape[1])]

shap_matrix = pd.DataFrame(
    shap_array,
    columns=feature_names,
    index=X_test.index
)

# Attach customer IDs using the original dataframe index. Use df_clean when it
# exists, otherwise fall back to the cleaned df loaded earlier in the notebook.
id_source = globals().get("df_clean", globals().get("df"))
if id_source is None:
    raise NameError("Neither 'df_clean' nor 'df' is available for customer IDs.")
id_column = next(
    (column for column in ("customerID", "CustomerID") if column in id_source.columns),
    None
)
if id_column is None:
    raise KeyError("Could not find a customer ID column in the available dataframe.")

shap_matrix.insert(
    0,
    "CustomerID",
    id_source.loc[X_test.index, id_column].to_numpy()
)

# Get probabilities from the full pipeline/model and apply the saved threshold.
churn_probabilities = final_xgb_model.predict_proba(X_test)[:, 1]
threshold = model_bundle["threshold"]

shap_matrix["Churn_Probability"] = churn_probabilities
shap_matrix["Risk_Category"] = np.where(
    churn_probabilities >= threshold,
    "HIGH",
    "LOW"
)

print("SHAP matrix shape:", shap_matrix.shape)
display(shap_matrix.head())

In [49]:
required_variables = [
    "model_bundle",
    "final_xgb_model",
    "X_test",
    "X_train",
    "df_clean",
    "explainer",
    "shap_values"
]

for name in required_variables:
    print(f"{name}: {'Available' if name in globals() else 'Missing'}")

In [53]:
import joblib
import os

model_path = "models/churn_model_bundle.joblib"

print("Model file exists:", os.path.exists(model_path))

model_bundle = joblib.load(model_path)

final_xgb_model = model_bundle["model"]
optimal_threshold = model_bundle["threshold"]

print("\nModel loaded successfully!")
print("Model type:", type(final_xgb_model))
print("Threshold:", optimal_threshold)

print("\nPipeline steps:")
for name, step in final_xgb_model.named_steps.items():
    print(name, ":", type(step))

In [55]:
import shap
import pandas as pd
import numpy as np
from scipy import sparse

# Extract pipeline components
preprocessor = final_xgb_model.named_steps["preprocessor"]
xgb_model = final_xgb_model.named_steps["model"]

# Transform test data using the saved preprocessing pipeline
X_test_processed = preprocessor.transform(X_test)

# Get transformed feature names
feature_names = preprocessor.get_feature_names_out()

# Convert to dense array if necessary
if sparse.issparse(X_test_processed):
    X_test_processed = X_test_processed.toarray()

# Create a DataFrame with feature names
X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

# Create SHAP explainer
explainer = shap.TreeExplainer(xgb_model)

# Generate explanations
shap_values = explainer(X_test_processed_df)

print("SHAP explanations generated!")
print("Shape:", shap_values.values.shape)
print("Number of features:", len(shap_values.feature_names))

In [57]:
churn_probabilities = final_xgb_model.predict_proba(X_test)[:, 1]

print("Test customers:", len(X_test))
print("SHAP explanations:", len(shap_values.values))
print("Predictions:", len(churn_probabilities))

assert len(X_test) == len(shap_values.values)
assert len(X_test) == len(churn_probabilities)

print("\nAll three are aligned!")

In [59]:
import pandas as pd
import numpy as np

# Get transformed feature names
feature_names = list(shap_values.feature_names)

# Create SHAP contribution matrix
shap_matrix = pd.DataFrame(
    shap_values.values,
    columns=feature_names,
    index=X_test.index
)

# Add predictions
churn_probabilities = final_xgb_model.predict_proba(X_test)[:, 1]

shap_matrix["Churn_Probability"] = churn_probabilities

# Use the same three risk categories as our earlier analysis
shap_matrix["Risk_Category"] = np.select(
    [
        churn_probabilities >= 0.70,
        churn_probabilities >= 0.30
    ],
    [
        "HIGH",
        "MEDIUM"
    ],
    default="LOW"
)

print("SHAP analysis table:", shap_matrix.shape)

display(
    shap_matrix[
        ["Churn_Probability", "Risk_Category"]
    ].head(10)
)

In [60]:
def get_top_positive_features(row, top_n=3):

    contributions = row[feature_names]

    positive = contributions[
        contributions > 0
    ].sort_values(ascending=False)

    return [
        (feature, round(float(value), 4))
        for feature, value in positive.head(top_n).items()
    ]


def get_top_negative_features(row, top_n=3):

    contributions = row[feature_names]

    negative = contributions[
        contributions < 0
    ].sort_values()

    return [
        (feature, round(float(value), 4))
        for feature, value in negative.head(top_n).items()
    ]


shap_matrix["Top_Churn_Drivers"] = shap_matrix.apply(
    get_top_positive_features,
    axis=1
)

shap_matrix["Top_Protective_Factors"] = shap_matrix.apply(
    get_top_negative_features,
    axis=1
)

display(
    shap_matrix[
        [
            "Churn_Probability",
            "Risk_Category",
            "Top_Churn_Drivers",
            "Top_Protective_Factors"
        ]
    ].head(10)
)

In [62]:
print("Customer risk distribution:")

display(
    shap_matrix["Risk_Category"]
    .value_counts()
    .rename_axis("Risk_Category")
    .reset_index(name="Customer_Count")
)

print("\nAverage churn probability by risk category:")

display(
    shap_matrix.groupby("Risk_Category")[
        "Churn_Probability"
    ].mean().sort_values(ascending=False)
)

In [63]:
import pandas as pd
import numpy as np

df_clean = pd.read_csv("data/processed/cleaned_clients.csv")


# Reference monthly charges for identifying unusually high charges
monthly_charge_reference = X_train["MonthlyCharges"].median()

# Original customer records, retaining customer IDs
customer_records = df_clean.loc[X_test.index]


def generate_shap_recommendations(customer_id, customer, shap_row):

    recommendations = []

    def is_positive(feature):
        return (
            feature in shap_row.index
            and shap_row[feature] > 0
        )

    def add_recommendation(reason, action, priority, feature):
        recommendations.append({
            "CustomerID": customer_id,
            "Recommendation_Reason": reason,
            "Recommended_Action": action,
            "Priority": priority,
            "SHAP_Feature": feature,
            "SHAP_Contribution": round(
                float(shap_row[feature]), 4
            )
        })

    # 1. Contract
    feature = "cat__Contract_Month-to-month"

    if (
        customer.get("Contract") == "Month-to-month"
        and is_positive(feature)
    ):
        add_recommendation(
            "Month-to-month contract",
            "Present suitable annual or two-year contract options.",
            "High",
            feature
        )

    # 2. Customer tenure
    feature = "num__tenure"

    if (
        customer.get("tenure", 999) <= 12
        and is_positive(feature)
    ):
        add_recommendation(
            "Short customer tenure",
            "Provide onboarding assistance and schedule a satisfaction check-in.",
            "High",
            feature
        )

    # 3. Online security
    feature = "cat__OnlineSecurity_No"

    if (
        customer.get("OnlineSecurity") == "No"
        and is_positive(feature)
    ):
        add_recommendation(
            "No online security service",
            "Explain available online security services and offer a consultation.",
            "Medium",
            feature
        )

    # 4. Technical support
    feature = "cat__TechSupport_No"

    if (
        customer.get("TechSupport") == "No"
        and is_positive(feature)
    ):
        add_recommendation(
            "No technical support service",
            "Explain available technical support plans.",
            "Medium",
            feature
        )

    # 5. Electronic check payment
    feature = "cat__PaymentMethod_Electronic check"

    if (
        customer.get("PaymentMethod") == "Electronic check"
        and is_positive(feature)
    ):
        add_recommendation(
            "Electronic check payment method",
            "Discuss convenient automatic payment alternatives.",
            "Low",
            feature
        )

    # 6. Higher monthly charges
    feature = "num__MonthlyCharges"

    if (
        customer.get("MonthlyCharges", 0) >= monthly_charge_reference
        and is_positive(feature)
    ):
        add_recommendation(
            "Higher monthly charges",
            "Review the current plan and identify cost-effective alternatives.",
            "Medium",
            feature
        )

    return recommendations

In [65]:
recommendation_records = []

for idx, customer in customer_records.iterrows():

    customer_id = customer["customerID"]

    shap_row = shap_matrix.loc[idx, feature_names]

    customer_recommendations = generate_shap_recommendations(
        customer_id,
        customer,
        shap_row
    )

    recommendation_records.extend(customer_recommendations)


shap_recommendation_df = pd.DataFrame(
    recommendation_records,
    columns=[
        "CustomerID",
        "Recommendation_Reason",
        "Recommended_Action",
        "Priority",
        "SHAP_Feature",
        "SHAP_Contribution"
    ]
)

print("Total recommendation records:", len(shap_recommendation_df))

print(
    "Unique customers receiving recommendations:",
    shap_recommendation_df["CustomerID"].nunique()
    if not shap_recommendation_df.empty else 0
)

display(shap_recommendation_df.head(15))

In [66]:
if not shap_recommendation_df.empty:

    print("Recommendation distribution:")

    display(
        shap_recommendation_df[
            "Recommendation_Reason"
        ].value_counts().reset_index()
    )

    print("\nPriority distribution:")

    display(
        shap_recommendation_df[
            "Priority"
        ].value_counts().reset_index()
    )

else:
    print("No SHAP-supported recommendations generated.")

In [68]:
# Create a customer-level lookup table
customer_lookup = pd.DataFrame({
    "CustomerID": df_clean.loc[X_test.index, "customerID"].values,
    "Churn_Probability": churn_probabilities,
    "Risk_Category": shap_matrix["Risk_Category"].values
})

# Remove any previous risk columns before merging, if present
for col in ["Churn_Probability", "Risk_Category"]:
    if col in shap_recommendation_df.columns:
        shap_recommendation_df = shap_recommendation_df.drop(columns=col)

# Merge customer risk information into recommendations
shap_recommendation_df = shap_recommendation_df.merge(
    customer_lookup,
    on="CustomerID",
    how="left",
    validate="many_to_one"
)

# Assign action urgency
urgency_map = {
    "HIGH": "URGENT",
    "MEDIUM": "PLANNED",
    "LOW": "MONITOR"
}

shap_recommendation_df["Action_Urgency"] = (
    shap_recommendation_df["Risk_Category"].map(urgency_map)
)

display(shap_recommendation_df.head(15))

In [72]:
customer_lookup = pd.DataFrame({
    "CustomerID": df_clean.loc[X_test.index, "customerID"].values,
    "Churn_Probability": churn_probabilities,
    "Risk_Category": shap_matrix["Risk_Category"].values
})

shap_recommendation_df = shap_recommendation_df.merge(
    customer_lookup,
    on="CustomerID",
    how="left",
    validate="many_to_one"
)

# Assign urgency based on overall customer risk
urgency_map = {
    "HIGH": "URGENT",
    "MEDIUM": "PLANNED",
    "LOW": "MONITOR"
}

shap_recommendation_df["Action_Urgency"] = (
    shap_recommendation_df["Risk_Category"].map(urgency_map)
)

display(shap_recommendation_df.head(15))

In [74]:
print("Total recommendation records:",
      len(shap_recommendation_df))

print("Unique customers:",
      shap_recommendation_df["CustomerID"].nunique())

print("\nMissing values:")
print(
    shap_recommendation_df[
        ["Churn_Probability", "Risk_Category", "Action_Urgency"]
    ].isnull().sum()
)

print("\nRisk distribution:")
display(shap_recommendation_df["Risk_Category"].value_counts())

print("\nUrgency distribution:")
display(shap_recommendation_df["Action_Urgency"].value_counts())

assert shap_recommendation_df["Risk_Category"].notna().all()

print("Recommendation enrichment validated!")

In [76]:
# Check consistency of customer-level information

risk_consistency = (
    shap_recommendation_df
    .groupby("CustomerID")["Risk_Category"]
    .nunique()
    .max()
)

probability_consistency = (
    shap_recommendation_df
    .groupby("CustomerID")["Churn_Probability"]
    .nunique()
    .max()
)

print("Maximum risk categories per customer:", risk_consistency)
print("Maximum probability values per customer:", probability_consistency)

assert risk_consistency == 1
assert probability_consistency == 1

print("\nCustomer-level consistency check PASSED!")

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=2cb50308-81e6-4757-a2d6-155b4c4296dc' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>